In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# Import feature engineering utilities for lagging
from src.feature_utils import create_lagged_features

# import pyfredapi as pf
import pandas as pd
import stockstats
# from fred_api_key import FRED_API_KEY
# from time import sleep

# API_KEY = FRED_API_KEY

# Import technical indicator definitions
from index_ticker import SP500
from stockstats_technicals import STOCKSTATS_TECHNICALS

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [2]:
from pandas.tseries.holiday import USFederalHolidayCalendar
import pandas_market_calendars as mcal

# Federal holidays (for on_holiday)
cal = USFederalHolidayCalendar()

federal_holidays = cal.holidays(
    start=START_DATE,
    end=END_DATE
)

# NYSE closure holidays (for pre/post effects)
nyse = mcal.get_calendar("NYSE")

# Get actual NYSE trading days in the date range
nyse_schedule = nyse.schedule(
    start_date=START_DATE,
    end_date=END_DATE
)

nyse_trading_days = pd.DatetimeIndex(nyse_schedule.index)

# Find calendar days when NYSE was closed
all_days = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

nyse_holidays = all_days.difference(nyse_trading_days)

# Optional: keep only dates (remove time component if present)
nyse_holidays = pd.DatetimeIndex(nyse_holidays.normalize())

In [3]:
import yfinance as yf
# df_stock: indexed by trading dates
df_stock = yf.download(["^GSPC", "^DJI", "^IXIC", "^NDX", "^NYA", "^RUT",
                        # "DX-Y.NYB", "^FTSE", "^N225", "^GDAXI", "^STI", "^TWII", "000001.SS",
                        # "^FCHI", "^STOXX50E", "^GSPTSE", "^BVSP"
                        ],
                        start=START_DATE, end=END_DATE, auto_adjust=True)

df_ref = yf.download(["^VIX", "^VVIX", "DX-Y.NYB", "^FTSE", "^N225", "^GDAXI", "^STI",
                      "^TWII", "000001.SS", "^FCHI", "^STOXX50E", "^GSPTSE", "^BVSP",
                      ],
                      start="1999-12-20", end=END_DATE, auto_adjust=True)

# Sort first
# df_stock = df_stock.sort_index()
# df_ref = df_ref.sort_index()

# Build a master index containing every date appearing in either DataFrame
master_index = df_stock.index.union(df_ref.index)

# Expand df_ref onto the master index, then forward-fill
df_ref_aligned = (
    df_ref
    .reindex(master_index)
    .ffill()
    .loc[df_stock.index]
)

# Left join
df_stock = df_stock.join(df_ref_aligned, how="left")

# Day-of-week dummy variables (Monday=0, ..., Friday=4)
weekday = df_stock.index.dayofweek

df_stock["mon"] = (weekday == 0).astype("int8")
df_stock["tues"] = (weekday == 1).astype("int8")
df_stock["wed"] = (weekday == 2).astype("int8")
df_stock["thurs"] = (weekday == 3).astype("int8")
df_stock["fri"] = (weekday == 4).astype("int8")

# holiday_dates: DatetimeIndex from USFederalHolidayCalendar
df_stock["on_holiday"] = 0
df_stock["pre_holiday"] = 0
df_stock["post_holiday"] = 0

# Holidays that occur on trading days
df_stock.loc[df_stock.index.isin(federal_holidays), "on_holiday"] = 1

for h in federal_holidays:
    # Last trading day before holiday
    prev = df_stock.index[df_stock.index < h]
    if len(prev):
        df_stock.loc[prev[-1], "pre_holiday"] = 1

    # First trading day after holiday
    nxt = df_stock.index[df_stock.index > h]
    if len(nxt):
        df_stock.loc[nxt[0], "post_holiday"] = 1

# Percentage-return windows
return_windows = [1, 5, 10, 20, 60, 120]

# Rolling-volatility windows
vol_windows = [5, 10, 20, 60]

close = df_stock["Close"]

# Daily returns (required for volatility)
daily_ret = close.pct_change(fill_method=None)

for w in return_windows:
    ret = close.pct_change(w, fill_method=None)
    ret.columns = pd.MultiIndex.from_product(
        [[f"Ret_{w}"], ret.columns],
        names=df_stock.columns.names,
    )
    df_stock = df_stock.join(ret)

for w in vol_windows:
    vol = daily_ret.rolling(w).std()
    vol.columns = pd.MultiIndex.from_product(
        [[f"Vol_{w}"], vol.columns],
        names=df_stock.columns.names,
    )
    df_stock = df_stock.join(vol)

[*********************100%***********************]  5 of 6 completed
[*********************100%***********************]  12 of 13 completed


In [4]:
# ============================================================================
# CREATE LAGGED VERSIONS OF ROLLING FEATURES
# ============================================================================
# LAGGING RATIONALE:
# Rolling returns (Ret_*) and rolling volatility (Vol_*) include data through
# the close of trading day T. Predictions are made BEFORE the US market opens
# on day T, so day T's data is not yet available. We lag by 1 trading day so
# the features used at prediction time only include information through T-1.
#
# Example:
#   Original: Ret_5 on day T = return from day T-4 to day T (includes T's close)
#   Lagged:   Ret_5_lag1 on day T = Ret_5 value from day T-1 (only through T-1's close)
#
# Calendar features (day-of-week dummies, holidays, FOMC) are NOT lagged
# because they are determined in advance and known before market open.
# ============================================================================

# Collect all rolling feature column names to lag
rolling_cols_to_lag = []

# All Ret_* columns
for w in return_windows:
    for col in close.columns:
        rolling_cols_to_lag.append((f"Ret_{w}", col))

# All Vol_* columns
for w in vol_windows:
    for col in daily_ret.columns:
        rolling_cols_to_lag.append((f"Vol_{w}", col))

# Create lagged versions using feature_utils
df_lagged = create_lagged_features(
    df_stock,
    rolling_cols_to_lag,
    lag_period=1,
    multiindex=True,
)

# Add lagged columns to df_stock
for col in rolling_cols_to_lag:
    lagged_col = (f"{col[0]}_lag1", col[1])
    df_stock[lagged_col] = df_lagged[lagged_col]

print(f"✓ Created {len(rolling_cols_to_lag)} lagged rolling feature columns")
print(f"  Sample lagged columns: {list(rolling_cols_to_lag[:3])} → {[c for c in df_stock.columns if '_lag1' in str(c)][:3]}")

c:\Users\jackshang\Desktop\Projects\sp500-prediction\src\feature_utils.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[lagged_col_name] = df[col].shift(lag_period)
c:\Users\jackshang\Desktop\Projects\sp500-prediction\src\feature_utils.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[lagged_col_name] = df[col].shift(lag_period)
c:\Users\jackshang\Desktop\Projects\sp500-prediction\src\feature_utils.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.ins

✓ Created 190 lagged rolling feature columns
  Sample lagged columns: [('Ret_1', '^DJI'), ('Ret_1', '^GSPC'), ('Ret_1', '^IXIC')] → [('Ret_1_lag1', '^DJI'), ('Ret_1_lag1', '^GSPC'), ('Ret_1_lag1', '^IXIC')]


C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\1698041165.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_stock[lagged_col] = df_lagged[lagged_col]
C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\1698041165.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_stock[lagged_col] = df_lagged[lagged_col]
C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\1698041165.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [5]:
# print(df_stock.head())
df_stock.to_csv(CACHE_PATH / "indices_from_2000.csv")

In [6]:
# ============================================================================
# LOAD FOMC CALENDAR AND CALCULATE DAYS_SINCE_FOMC
# ============================================================================
# LAGGING RATIONALE FOR FOMC FEATURES:
# FOMC meeting announcement dates are published well in advance. They are known
# before the US market opens on day T, so we do NOT lag this feature.
# ============================================================================

# Load FOMC calendar if available
fomc_path = CACHE_PATH / "fomc_calendar_2000_present.csv"

if fomc_path.exists():
    fomc = pd.read_csv(fomc_path)
    fomc["date"] = pd.to_datetime(fomc["date"])
    
    # Keep only announcement (decision) dates
    fomc_announcements = (
        fomc.loc[fomc["is_fomc_day"] == 1, ["date"]]
        .drop_duplicates()
        .rename(columns={"date": "last_fomc_date"})
        .sort_values("last_fomc_date")
    )
    
    # Trading dates from df_stock
    tmp = pd.DataFrame({"date": df_stock.index}).sort_values("date")
    
    # Match each trading day to the most recent FOMC announcement
    tmp = pd.merge_asof(
        tmp,
        fomc_announcements,
        left_on="date",
        right_on="last_fomc_date",
        direction="backward",
    )
    
    # Calendar days since last FOMC announcement
    tmp["days_since_fomc"] = (
        tmp["date"] - tmp["last_fomc_date"]
    ).dt.days
    
    # Add to df_stock (single-level column, empty second level for MultiIndex)
    df_stock[("days_since_fomc", "")] = (
        tmp.set_index("date")["days_since_fomc"]
    )
    
    print(f"✓ Loaded FOMC calendar and calculated days_since_fomc")
else:
    print(f"⚠ FOMC calendar not found at {fomc_path}. Skipping days_since_fomc.")

✓ Loaded FOMC calendar and calculated days_since_fomc


C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\798200684.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_stock[("days_since_fomc", "")] = (


In [ ]:
# ============================================================================
# CALCULATE STOCKSTATS TECHNICAL INDICATORS FOR S&P 500
# ============================================================================
# LAGGING RATIONALE FOR TECHNICAL INDICATORS:
# All 54 StockStats technical indicators are computed using OHLCV data available
# in the input DataFrame. Since the input contains data through trading day T's
# close, the indicators at row T include T's data. However, predictions are made
# BEFORE the US market opens on day T, so day T's data is not yet available.
#
# We create lagged versions so that indicators used for prediction on day T only
# include information through day T-1's close.
#
# Example:
#   Original: RSI_14 on day T = computed using OHLCV through day T
#   Lagged:   RSI_14_lag1 on day T = RSI_14 value from day T-1
# ============================================================================

# Extract S&P 500 OHLCV data for technical indicator calculation
sp = pd.DataFrame({
    "open": df_stock[("Open", SP500)],
    "high": df_stock[("High", SP500)],
    "low": df_stock[("Low", SP500)],
    "close": df_stock[("Close", SP500)],
    "volume": df_stock[("Volume", SP500)],
})

# Convert to StockDataFrame for technical indicator calculation
sp = stockstats.StockDataFrame.retype(sp)

# Calculate technical indicators
technical_features = []

for feature in STOCKSTATS_TECHNICALS:
    try:
        # Add to df_stock with MultiIndex structure (Technical, indicator_name)
        df_stock[("Technical", feature)] = sp[feature]
        technical_features.append(feature)
    except Exception as e:
        print(f"⚠ Skipping {feature}: {e}")

print(f"✓ Calculated {len(technical_features)} StockStats technical indicators for S&P 500")

# Create lagged versions of all technical indicators
# LAGGING RATIONALE: See docstring above
technical_cols_to_lag = [("Technical", feat) for feat in technical_features]

df_lagged = create_lagged_features(
    df_stock,
    technical_cols_to_lag,
    lag_period=1,
    multiindex=True,
)

# Add lagged technical columns to df_stock
for col in technical_cols_to_lag:
    lagged_col = (f"{col[0]}_lag1", col[1])
    if lagged_col not in df_stock.columns:  # Avoid duplicate if already exists
        df_stock[lagged_col] = df_lagged[lagged_col]

print(f"✓ Created lagged versions of {len(technical_features)} technical indicators")
print("  Lagged columns use naming convention: ({feature_name}_lag1, '')")

C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\1376576956.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_stock[("Technical", feature)] = sp[feature]
C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\1376576956.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_stock[("Technical", feature)] = sp[feature]
C:\Users\jackshang\AppData\Local\Temp\ipykernel_16080\1376576956.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

✓ Calculated 51 StockStats technical indicators for S&P 500
✓ Created lagged versions of 51 technical indicators


NameError: name 'feature_name' is not defined